# Time-dependent toy example

Two-timestep electricity dispatch with a battery, demonstrating `pulpo.utils.time_extension`.

Activities: `solar`, `coal`, `battery_charge`, `battery_discharge`. Demand is placed on a virtual `electricity` product formed by PULPO `choices` over `{solar, coal, battery_discharge}`.

The storage spec `(battery_charge, battery_charge, K)` says: charging at *t-1* makes `K` units of stored energy available at *t*.

In [ ]:
import bw2data as bd

from pulpo import pulpo as pulpo_mod
from pulpo.datasets.elec_time_database import (
    setup_elec_time_db,
    PROJECT_NAME, DB_NAME,
    SOLAR_KEY, COAL_KEY, BATTERY_CHARGE_KEY, BATTERY_DISCHARGE_KEY,
)

setup_elec_time_db()
bd.projects.set_current(PROJECT_NAME)
db = bd.Database(DB_NAME)

solar     = next(a for a in db if a.key == SOLAR_KEY)
coal      = next(a for a in db if a.key == COAL_KEY)
charge    = next(a for a in db if a.key == BATTERY_CHARGE_KEY)
discharge = next(a for a in db if a.key == BATTERY_DISCHARGE_KEY)

## Define the time-dependent problem

At *t=0* solar is up to 200 kWh, but discharge is locked (the battery is empty). At *t=1* there is no sun and discharge is unlocked. Demand is 50 kWh in both timesteps. Battery round-trip efficiency: 90%.

In [ ]:
GWP100 = str(("GWP", "100a"))

time_steps = [0, 1]

choices = {
    "electricity": {solar: 1e6, coal: 1e6, discharge: 1e6},
}

# Per-timestep capacities. The dict can be either time-indexed
# ({t: {act: cap}}) or static ({act: cap}, broadcast across all timesteps).
upper_limit = {
    0: {solar: 200.0, coal: 1e6, charge: 1e6, discharge: 0.0},
    1: {solar:   0.0, coal: 1e6, charge:  0.0, discharge: 1e6},
}

demand = {
    0: {"electricity": 50.0},
    1: {"electricity": 50.0},
}

# Storage: charging at t-1 carries over with 90% efficiency.
storage = [(charge, charge, 0.9)]

## Solve with battery

In [ ]:
worker = pulpo_mod.PulpoOptimizer(PROJECT_NAME, DB_NAME, {GWP100: 1}, ".")
worker.get_lci_data()
worker.instantiate(
    choices=choices,
    demand=demand,
    upper_limit=upper_limit,
    time_steps=time_steps,
    storage=storage,
)
worker.solve()

inst = worker.instance
pmap = worker.lci_data["process_map"]
total_co2 = sum(inst.impacts[t, GWP100].value for t in time_steps)
print(f"Total CO2 with battery: {total_co2:.2f} kg")
for t in time_steps:
    print(
        f"  t={t}:  "
        f"solar={inst.scaling_vector[t, pmap[solar.key]].value:7.2f}  "
        f"coal={inst.scaling_vector[t, pmap[coal.key]].value:7.2f}  "
        f"charge={inst.scaling_vector[t, pmap[charge.key]].value:7.2f}  "
        f"discharge={inst.scaling_vector[t, pmap[discharge.key]].value:7.2f}"
    )

## Baseline: no storage

Without the battery the system has to fall back to coal at *t=1*, so total CO2 equals the *t=1* demand.

In [ ]:
worker_nb = pulpo_mod.PulpoOptimizer(PROJECT_NAME, DB_NAME, {GWP100: 1}, ".")
worker_nb.get_lci_data()
worker_nb.instantiate(
    choices=choices,
    demand=demand,
    upper_limit=upper_limit,
    time_steps=time_steps,
    storage=None,  # disable carry-over
)
worker_nb.solve()
inst_nb = worker_nb.instance
total_co2_nb = sum(inst_nb.impacts[t, GWP100].value for t in time_steps)
print(f"Total CO2 without battery: {total_co2_nb:.2f} kg")

## Aggregated impact bound

`upper_imp_agg_limit` constrains the **sum** of an indicator across all timesteps -- e.g. an annual CO2 budget independent of per-step limits.

In [ ]:
worker_cap = pulpo_mod.PulpoOptimizer(PROJECT_NAME, DB_NAME, {GWP100: 1}, ".")
worker_cap.get_lci_data()
worker_cap.instantiate(
    choices=choices,
    demand=demand,
    upper_limit=upper_limit,
    time_steps=time_steps,
    storage=storage,
    upper_imp_agg_limit={GWP100: 10.0},  # 10 kg CO2 cap over the horizon
)
worker_cap.solve()
inst_cap = worker_cap.instance
print("Total CO2 with budget:",
      sum(inst_cap.impacts[t, GWP100].value for t in time_steps))